In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="07-event-recommendation",
    notebook_name="03_ranking_and_personalization.ipynb"
)

# Ranking, Personalization, Diversity, and Notification Optimization

## For a 12-Year-Old

You know how when you open YouTube, it shows you videos it thinks you will like? And sometimes it is great -- but sometimes it shows you the SAME TYPE of video over and over. Like, yes, I watched one cat video, but I do not ONLY want cat videos!

Event recommendation has the same problem. If you went to a jazz concert once, should we ONLY show you jazz events? Probably not. You might also love art shows, tech meetups, or food festivals. This notebook covers:

1. **How to rank events** (which one goes on top?)
2. **How to make it personal** (customized to YOUR taste)
3. **How to keep it diverse** (not just the same category)
4. **When to send notifications** (not too many, not too few)

---

## Staff-Level Summary

This notebook covers the ranking and personalization layer of the event recommendation system:
- Learning to Rank approaches (pointwise, pairwise, listwise)
- Model architecture progression (LR -> Decision Trees -> GBDT -> Neural Networks)
- Diversity via Maximal Marginal Relevance (MMR) and category-aware re-ranking
- Notification optimization to maximize engagement without fatigue

## 1. Personalization Strategies

### The Personalization Pyramid

```
              /\
             /  \          Level 4: DEEP PERSONALIZATION
            / NN \         Neural network with embeddings,
           / with \        continual learning, real-time features
          / embeds \       
         /----------\      Level 3: FEATURE-BASED
        /   GBDT    \      Gradient-boosted trees with
       / with rich   \     hand-crafted features
      /  features     \
     /----------------\    Level 2: COLLABORATIVE
    / Content-Based &  \   "Users who liked X also liked Y"
   / Collaborative      \  (limited for ephemeral events)
  /--------------------\
 /   Popular & Nearby   \  Level 1: POPULARITY
/________________________\ "Show popular events near you"
```

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# === Level 1: Popularity-Based (Baseline) ===

def popularity_recommendation(events_df, user_city, top_k=5):
    """
    Simplest recommendation: popular events in user's city.
    
    12-year-old version:
    'What are the most popular events near me?'
    Like sorting by 'most tickets sold' on Eventbrite.
    
    Pros: Simple, no cold-start problem, works for new users
    Cons: Not personalized at all -- everyone sees the same list
    """
    # Filter by city
    local = events_df[events_df['city'] == user_city].copy()
    # Sort by registration count
    local = local.sort_values('num_registered', ascending=False)
    return local.head(top_k)


# Sample events with registration counts
events_data = pd.DataFrame({
    'event_id': range(1, 11),
    'title': [
        'Dua Lipa Concert', 'Jazz Night', 'AI Meetup', 'Wine Festival',
        'Comedy Show', 'Basketball Game', 'Art Exhibition', 'Book Club',
        'Yoga in the Park', 'Food Truck Rally'
    ],
    'category': [
        'Music', 'Music', 'Tech', 'Food', 'Comedy',
        'Sports', 'Art', 'Education', 'Wellness', 'Food'
    ],
    'city': ['Miami']*10,
    'num_registered': [500, 120, 80, 300, 200, 450, 60, 30, 150, 350],
    'price': [200, 50, 0, 75, 40, 140, 25, 0, 15, 10],
    'description': [
        'Pop music concert with international star',
        'Live jazz ensemble at downtown venue',
        'Workshop on large language models and AI',
        'Taste wines from around the world',
        'Stand-up comedy night with top comedians',
        'NBA basketball game heat vs celtics',
        'Modern art exhibition featuring local artists',
        'Monthly book discussion group',
        'Morning yoga session in Bayfront Park',
        'Food trucks gathering with live music'
    ]
})

print("=== Level 1: Popularity-Based Recommendation ===")
popular = popularity_recommendation(events_data, 'Miami')
for i, (_, event) in enumerate(popular.iterrows()):
    print(f"  #{i+1}: {event['title']} ({event['category']}) - {event['num_registered']} registered")

print("\nProblem: This is the SAME for every user. A jazz lover and a sports")
print("fan see the exact same list. Not personalized at all!")

In [ ]:
# === Level 2: Content-Based Personalization ===

def content_based_recommendation(events_df, user_past_descriptions, top_k=5):
    """
    Recommend events similar to what the user attended before.
    
    12-year-old version:
    'You went to a jazz concert and a blues night before.
    Here are more music events that sound similar!'
    
    Uses TF-IDF to convert descriptions to vectors, then cosine similarity.
    """
    # Combine user's past event descriptions into a "user profile"
    user_profile = " ".join(user_past_descriptions)
    
    # Create TF-IDF vectors
    all_descriptions = events_df['description'].tolist() + [user_profile]
    vectorizer = TfidfVectorizer(stop_words='english')
    tfidf_matrix = vectorizer.fit_transform(all_descriptions)
    
    # Compute similarity between user profile and each event
    user_vector = tfidf_matrix[-1]
    event_vectors = tfidf_matrix[:-1]
    similarities = cosine_similarity(user_vector, event_vectors).flatten()
    
    # Add similarity scores to events
    events_copy = events_df.copy()
    events_copy['description_similarity'] = similarities
    events_copy = events_copy.sort_values('description_similarity', ascending=False)
    
    return events_copy.head(top_k)


# User who loves music events
music_lover_past = [
    "Amazing live jazz performance at the club",
    "Rock concert with incredible guitar solos",
    "Live music festival with multiple stages"
]

# User who loves tech and learning
tech_lover_past = [
    "Machine learning workshop with hands-on coding",
    "AI conference discussing latest research papers",
    "Python programming meetup and code review"
]

print("=== Level 2: Content-Based Personalization ===")
print("\n--- For Music Lover ---")
recs = content_based_recommendation(events_data, music_lover_past)
for i, (_, event) in enumerate(recs.iterrows()):
    print(f"  #{i+1}: {event['title']} (similarity: {event['description_similarity']:.3f})")

print("\n--- For Tech Lover ---")
recs = content_based_recommendation(events_data, tech_lover_past)
for i, (_, event) in enumerate(recs.iterrows()):
    print(f"  #{i+1}: {event['title']} (similarity: {event['description_similarity']:.3f})")

print("\nNotice: Different users get different rankings based on their history!")
print("This is the description_similarity feature from our feature engineering.")

In [ ]:
# === Level 3 & 4: Full Feature-Based Ranking (GBDT and NN) ===

# This is the production approach from the PDF.
# Combine ALL feature categories into a single prediction.

from sklearn.ensemble import GradientBoostingClassifier

def simulate_full_ranking():
    """
    Simulate the complete ranking pipeline with all feature categories.
    
    Staff-level detail:
    The pointwise LTR model takes ALL features as input and predicts
    P(register | user, event). Events are ranked by this probability.
    
    Feature categories:
    1. Location: distance, walk score, same city/country, + similarities
    2. Time: remaining time, day/hour match, travel time, + similarities
    3. Social: friends registered, host is friend, invitations, popularity
    4. User: age, gender
    5. Event: price, description similarity
    """
    np.random.seed(42)
    n_events = 10  # Candidate events after filtering
    
    # Simulate features for 10 candidate events (for one specific user)
    features = {
        'event': [f'Event_{i}' for i in range(n_events)],
        # Location features
        'distance_bucket': [0, 1, 2, 0, 3, 1, 0, 4, 1, 0],
        'same_city': [1, 1, 1, 1, 0, 1, 1, 0, 1, 1],
        'walk_score_sim': [2, 5, 10, 3, 25, 8, 1, 30, 6, 4],
        # Time features
        'remaining_time_bucket': [6, 7, 5, 6, 8, 7, 5, 8, 6, 7],
        'day_match': [0.4, 0.1, 0.3, 0.4, 0.0, 0.1, 0.3, 0.0, 0.4, 0.1],
        'hour_match': [0.3, 0.0, 0.2, 0.3, 0.0, 0.1, 0.2, 0.0, 0.3, 0.1],
        # Social features
        'num_friends_registered': [3, 0, 1, 2, 0, 0, 1, 0, 4, 0],
        'host_is_friend': [1, 0, 0, 1, 0, 0, 0, 0, 1, 0],
        'num_registered': [200, 50, 80, 150, 30, 100, 40, 20, 300, 60],
        # Event features
        'price_bucket': [2, 1, 0, 1, 3, 1, 0, 2, 1, 0],
        'description_sim': [0.8, 0.2, 0.6, 0.7, 0.1, 0.3, 0.5, 0.1, 0.9, 0.2],
    }
    
    df = pd.DataFrame(features)
    feature_cols = [c for c in df.columns if c != 'event']
    
    # Simulate model predictions (in production, this comes from trained model)
    # Higher score = more likely to register
    scores = (
        0.3 * (1 - df['distance_bucket'] / 5) +        # Closer = better
        0.15 * df['same_city'] +                        # Same city = better
        0.1 * (1 - df['walk_score_sim'] / 30) +         # Similar walkability = better
        0.1 * df['day_match'] +                         # Day preference match
        0.15 * df['num_friends_registered'] / 5 +       # Friends going = better
        0.1 * df['host_is_friend'] +                    # Host is friend = better
        0.1 * df['description_sim']                     # Content match
    ) + np.random.normal(0, 0.02, n_events)  # Small noise
    
    df['predicted_score'] = scores
    df = df.sort_values('predicted_score', ascending=False)
    
    print("=== Full Feature-Based Ranking ===")
    print(f"{'Rank':>4} {'Event':<10} {'Score':>6} {'Dist':>5} {'City':>5} {'Friends':>8} {'Host_Fr':>8} {'Desc_Sim':>9}")
    print("-" * 65)
    for rank, (_, row) in enumerate(df.iterrows(), 1):
        print(f"{rank:>4} {row['event']:<10} {row['predicted_score']:.3f} "
              f"{row['distance_bucket']:>5} {row['same_city']:>5} "
              f"{row['num_friends_registered']:>8} {row['host_is_friend']:>8} "
              f"{row['description_sim']:>9.2f}")
    
    print("\nNotice: Top events tend to be close, in same city, have friends going,")
    print("and match user's content preferences. The model combines ALL signals.")
    
    return df


ranked_events = simulate_full_ranking()

## 2. Learning to Rank: Deep Dive

### The Three Approaches

In [ ]:
import torch
import torch.nn as nn

# === Pointwise LTR ===
# This is what we use: predict relevance for each item independently

class PointwiseRanker(nn.Module):
    """
    Pointwise Learning-to-Rank model.
    
    12-year-old version:
    Look at each event one at a time and ask:
    'Would Alice sign up for THIS event? Yes (1) or No (0)?'
    Then sort all events by their 'yes' probability.
    
    Staff-level detail:
    - Binary classification per (user, event) pair
    - Loss: Binary Cross Entropy
    - Score is P(register | user, event)
    - Final ranking: sort by score descending
    - Limitation: scores are independent -- no relative comparison
    """
    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.network(x).squeeze(-1)


# === Pairwise LTR (RankNet-style) ===

class PairwiseRanker(nn.Module):
    """
    Pairwise Learning-to-Rank model (RankNet approach).
    
    12-year-old version:
    Hold up TWO event flyers and ask:
    'Which one would Alice prefer -- Event A or Event B?'
    Do this for many pairs, and piece together the full ranking.
    
    Staff-level detail:
    - Learns a scoring function s(x)
    - For a pair (i, j) where i is more relevant:
      P(i > j) = sigmoid(s(i) - s(j))
    - Loss: Cross-entropy on pairwise preferences
    - More accurate than pointwise but O(n^2) pairs
    """
    def __init__(self, input_dim):
        super().__init__()
        self.scorer = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
    
    def forward(self, x):
        return self.scorer(x).squeeze(-1)
    
    def compute_pairwise_loss(self, x_i, x_j):
        """
        x_i: features of the more relevant item
        x_j: features of the less relevant item
        """
        s_i = self.scorer(x_i).squeeze(-1)
        s_j = self.scorer(x_j).squeeze(-1)
        # P(i > j) = sigmoid(s_i - s_j)
        # We want this to be close to 1
        return -torch.mean(torch.log(torch.sigmoid(s_i - s_j) + 1e-8))


# Demonstrate both
input_dim = 20

# Pointwise
pointwise = PointwiseRanker(input_dim)
sample_features = torch.randn(5, input_dim)  # 5 events
pointwise_scores = pointwise(sample_features)

print("=== Pointwise LTR ===")
print(f"Input: 5 events with {input_dim} features each")
print(f"Output: Independent probability for each event")
for i, score in enumerate(pointwise_scores.detach().numpy()):
    print(f"  Event {i}: P(register) = {score:.4f}")
print(f"Ranking: {np.argsort(-pointwise_scores.detach().numpy()).tolist()}")

print("\n=== Pairwise LTR ===")
pairwise = PairwiseRanker(input_dim)
pairwise_scores = pairwise(sample_features)
print(f"Input: Same 5 events")
print(f"Scores (not probabilities -- used for relative comparison):")
for i, score in enumerate(pairwise_scores.detach().numpy()):
    print(f"  Event {i}: score = {score:.4f}")

# Show pairwise comparison
i, j = 0, 1
p_ij = torch.sigmoid(pairwise_scores[i] - pairwise_scores[j]).item()
print(f"\nP(Event {i} > Event {j}) = {p_ij:.4f}")
print(f"Ranking: {np.argsort(-pairwise_scores.detach().numpy()).tolist()}")

print("\n=== Why We Choose Pointwise ===")
print("1. Simpler to implement and debug")
print("2. Can use any binary classifier (LR, GBDT, NN)")
print("3. Pairwise/listwise are more accurate but harder to train")
print("4. Good enough for initial production system")

## 3. Diversity and Exploration

### The Filter Bubble Problem

If a user went to 3 jazz events, the model will learn to rank jazz events highly. But showing ONLY jazz events is bad because:
1. The user might also enjoy other things they have not tried
2. It gets boring (user fatigue)
3. The model never gets data about the user's other interests (exploration-exploitation tradeoff)

In [ ]:
# === Maximal Marginal Relevance (MMR) ===

def mmr_rerank(events, scores, categories, lambda_param=0.5, top_k=5):
    """
    Re-rank events using Maximal Marginal Relevance to ensure diversity.
    
    12-year-old version:
    Instead of just showing the top-5 highest-scored events (which might
    all be jazz events), we pick events one by one:
    1. Start with the highest-scored event
    2. For the next pick, choose the event that is both:
       - High scored (relevant to you)
       - DIFFERENT from what we already picked (diverse)
    3. Repeat until we have our top-k list
    
    It is like making a playlist -- you want good songs, but not all
    the same genre.
    
    Staff-level detail:
    MMR(e) = lambda * relevance(e) - (1-lambda) * max_similarity(e, selected)
    
    lambda = 1.0: pure relevance (no diversity)
    lambda = 0.0: pure diversity (ignore relevance)
    lambda = 0.5: balanced
    """
    selected = []
    remaining = list(range(len(events)))
    
    for _ in range(min(top_k, len(events))):
        best_idx = None
        best_mmr = -float('inf')
        
        for idx in remaining:
            relevance = scores[idx]
            
            # Compute similarity to already selected events
            if selected:
                # Simple category-based diversity: 1 if same category, 0 otherwise
                max_sim = max(
                    1.0 if categories[idx] == categories[s] else 0.0
                    for s in selected
                )
            else:
                max_sim = 0.0
            
            mmr_score = lambda_param * relevance - (1 - lambda_param) * max_sim
            
            if mmr_score > best_mmr:
                best_mmr = mmr_score
                best_idx = idx
        
        selected.append(best_idx)
        remaining.remove(best_idx)
    
    return selected


# Example: 10 events, many in the same category
events = [
    'Jazz Night', 'Jazz Festival', 'Jazz Brunch', 'AI Meetup', 'Wine Tasting',
    'Jazz Workshop', 'Basketball Game', 'Comedy Show', 'Jazz Improv', 'Art Gallery'
]
categories = [
    'Music', 'Music', 'Music', 'Tech', 'Food',
    'Music', 'Sports', 'Comedy', 'Music', 'Art'
]
scores = [0.95, 0.90, 0.85, 0.80, 0.75, 0.70, 0.65, 0.60, 0.55, 0.50]

# Pure relevance ranking
print("=== Pure Relevance Ranking (No Diversity) ===")
pure_ranking = np.argsort(-np.array(scores))[:5]
for i, idx in enumerate(pure_ranking):
    print(f"  #{i+1}: {events[idx]} ({categories[idx]}) - Score: {scores[idx]:.2f}")
print("  Problem: 4 out of 5 are Jazz (Music)! Boring!")

# MMR with diversity
print("\n=== MMR Ranking (lambda=0.5, Balanced) ===")
mmr_ranking = mmr_rerank(events, scores, categories, lambda_param=0.5, top_k=5)
for i, idx in enumerate(mmr_ranking):
    print(f"  #{i+1}: {events[idx]} ({categories[idx]}) - Score: {scores[idx]:.2f}")
print("  Better! Mix of categories while still showing relevant events.")

# MMR with high diversity
print("\n=== MMR Ranking (lambda=0.3, High Diversity) ===")
mmr_ranking_diverse = mmr_rerank(events, scores, categories, lambda_param=0.3, top_k=5)
for i, idx in enumerate(mmr_ranking_diverse):
    print(f"  #{i+1}: {events[idx]} ({categories[idx]}) - Score: {scores[idx]:.2f}")
print("  Even more diverse, but may sacrifice some relevance.")

In [ ]:
# === Category-Aware Re-Ranking ===

def category_aware_rerank(events, scores, categories, max_per_category=2, top_k=5):
    """
    Re-rank with a cap on events per category.
    
    12-year-old version:
    Rule: 'Show at most 2 events from the same category.'
    This way, even if jazz is your #1 love, you will see
    other types of events too.
    
    Staff-level detail:
    Simple but effective approach used in production systems.
    Can be combined with MMR for even better diversity.
    """
    sorted_indices = np.argsort(-np.array(scores))
    category_counts = {}
    selected = []
    
    for idx in sorted_indices:
        cat = categories[idx]
        count = category_counts.get(cat, 0)
        
        if count < max_per_category:
            selected.append(idx)
            category_counts[cat] = count + 1
        
        if len(selected) >= top_k:
            break
    
    return selected


print("=== Category-Aware Re-Ranking (max 2 per category) ===")
cat_ranking = category_aware_rerank(events, scores, categories, max_per_category=2, top_k=5)
for i, idx in enumerate(cat_ranking):
    print(f"  #{i+1}: {events[idx]} ({categories[idx]}) - Score: {scores[idx]:.2f}")
print("\nAt most 2 Music events, then other categories fill in.")

In [ ]:
# === Exploration-Exploitation Tradeoff ===

def epsilon_greedy_recommendation(events, scores, categories, epsilon=0.1, top_k=5):
    """
    Add exploration by occasionally showing random events.
    
    12-year-old version:
    90% of the time, show the events we THINK you will like (exploitation).
    10% of the time, show a RANDOM event (exploration).
    
    Why? Because maybe you would LOVE art shows, but we will never
    know unless we show you one! The random picks help us discover
    new things you might enjoy.
    
    Staff-level detail:
    Epsilon-greedy is the simplest exploration strategy.
    More sophisticated approaches:
    - Thompson Sampling
    - Upper Confidence Bound (UCB)
    - Contextual bandits
    """
    sorted_indices = np.argsort(-np.array(scores)).tolist()
    selected = []
    
    for _ in range(top_k):
        if np.random.random() < epsilon and sorted_indices:
            # Explore: pick a random event
            idx = np.random.choice(sorted_indices)
            selected.append((idx, 'EXPLORE'))
        else:
            # Exploit: pick the highest-scored remaining event
            idx = sorted_indices[0]
            selected.append((idx, 'EXPLOIT'))
        
        sorted_indices.remove(idx)
    
    return selected


np.random.seed(42)
print("=== Epsilon-Greedy Exploration (epsilon=0.1) ===")
for trial in range(3):
    print(f"\nTrial {trial + 1}:")
    result = epsilon_greedy_recommendation(events, scores, categories)
    for i, (idx, strategy) in enumerate(result):
        marker = " <-- RANDOM PICK!" if strategy == 'EXPLORE' else ""
        print(f"  #{i+1}: {events[idx]} ({categories[idx]}) - Score: {scores[idx]:.2f} [{strategy}]{marker}")

print("\nThe random picks help discover user interests outside their bubble.")
print("If a random pick gets a click/registration, we learn something new!")

## 4. Notification Optimization

### When to Notify Users

Notifications are powerful but dangerous. Too many = user disables them. Too few = missed opportunities.

In [ ]:
# === Notification Optimization System ===

class NotificationOptimizer:
    """
    Decide when and what to notify users about events.
    
    12-year-old version:
    Your phone should nudge you about a cool event, but:
    - Not at 3am (bad timing)
    - Not every 5 minutes (annoying)
    - Only for events you would actually care about (relevant)
    - Maybe when your friend just signed up (social trigger)
    
    Staff-level detail:
    Notification optimization is itself a ranking/decision problem:
    - What to notify about? (event relevance threshold)
    - When to send? (user's active hours, not too frequent)
    - What trigger? (new event, friend signed up, price drop, limited seats)
    - Budget: max N notifications per day per user
    """
    
    def __init__(self, max_daily_notifications=3, relevance_threshold=0.7):
        self.max_daily = max_daily_notifications
        self.threshold = relevance_threshold
        self.daily_count = {}  # user_id -> count today
    
    def should_notify(self, user_id, event_score, trigger_type, current_hour):
        """
        Decide whether to send a notification.
        
        Returns: (should_send, reason)
        """
        count = self.daily_count.get(user_id, 0)
        
        # Check daily budget
        if count >= self.max_daily:
            return False, f"Budget exceeded ({count}/{self.max_daily} sent today)"
        
        # Check quiet hours (no notifications 11pm - 8am)
        if current_hour >= 23 or current_hour < 8:
            return False, f"Quiet hours ({current_hour}:00)"
        
        # Check relevance threshold
        if event_score < self.threshold:
            return False, f"Below relevance threshold ({event_score:.2f} < {self.threshold})"
        
        # Social triggers get a boost (lower threshold)
        if trigger_type == 'friend_registered' and event_score >= self.threshold * 0.8:
            self.daily_count[user_id] = count + 1
            return True, f"Friend registered + decent relevance ({event_score:.2f})"
        
        # High-urgency triggers (limited seats, event tomorrow)
        if trigger_type == 'limited_seats' and event_score >= self.threshold * 0.9:
            self.daily_count[user_id] = count + 1
            return True, f"Limited seats + good relevance ({event_score:.2f})"
        
        # Standard notification
        if event_score >= self.threshold:
            self.daily_count[user_id] = count + 1
            return True, f"High relevance ({event_score:.2f})"
        
        return False, "No trigger matched"


# Simulate notification decisions
optimizer = NotificationOptimizer(max_daily_notifications=3, relevance_threshold=0.7)

notification_scenarios = [
    ("Alice", 1, 0.95, "new_event", 10, "Dua Lipa Concert"),
    ("Alice", 1, 0.80, "friend_registered", 14, "Jazz Night"),
    ("Alice", 1, 0.60, "new_event", 16, "Book Club"),
    ("Alice", 1, 0.85, "limited_seats", 19, "Comedy Show"),
    ("Alice", 1, 0.90, "new_event", 21, "Art Gallery"),  # Over budget
    ("Bob", 2, 0.75, "new_event", 3, "Late Night Jazz"),   # Quiet hours
    ("Bob", 2, 0.40, "new_event", 12, "Random Workshop"),  # Low relevance
    ("Bob", 2, 0.55, "friend_registered", 15, "Food Festival"),  # Friend boost!
]

print("=== Notification Optimization Results ===")
print(f"{'User':<8} {'Hour':>4} {'Score':>6} {'Trigger':<20} {'Event':<20} {'Send?':>6} Reason")
print("-" * 100)

for name, uid, score, trigger, hour, event in notification_scenarios:
    should_send, reason = optimizer.should_notify(uid, score, trigger, hour)
    send_str = "YES" if should_send else "NO"
    print(f"{name:<8} {hour:>4} {score:>6.2f} {trigger:<20} {event:<20} {send_str:>6} {reason}")

In [ ]:
# === Notification Trigger Types ===

print("=== Notification Triggers (Interview Discussion) ===")
print()

triggers = {
    "New High-Relevance Event": {
        "when": "New event matches user's profile strongly (score > threshold)",
        "message": "'A new jazz concert was just posted near you!'",
        "urgency": "Medium"
    },
    "Friend Registered": {
        "when": "User's friend signs up for an event the user might like",
        "message": "'Your friend Alice just signed up for Jazz Night!'",
        "urgency": "High (social proof is powerful)"
    },
    "Limited Seats": {
        "when": "Event is filling up, user previously showed interest",
        "message": "'Only 5 seats left for the Comedy Show you bookmarked!'",
        "urgency": "High (scarcity)"
    },
    "Event Tomorrow": {
        "when": "High-relevance event happens tomorrow, user has not registered",
        "message": "'Wine Festival is TOMORROW -- register before it is full!'",
        "urgency": "High (time pressure)"
    },
    "Price Drop": {
        "when": "Event user viewed drops in price",
        "message": "'The concert you viewed just dropped to $50!'",
        "urgency": "Medium"
    },
    "Personalized Digest": {
        "when": "Weekly summary of top recommendations",
        "message": "'Here are 5 events we think you will love this week'",
        "urgency": "Low (batch, not real-time)"
    }
}

for trigger, details in triggers.items():
    print(f"{trigger}")
    print(f"  When: {details['when']}")
    print(f"  Message: {details['message']}")
    print(f"  Urgency: {details['urgency']}")
    print()

## 5. The Two-Sided Marketplace Problem

### An Important Interview Talking Point

Event platforms are **two-sided marketplaces**:
- **Supply side**: Event hosts (create events, want attendees)
- **Demand side**: Users (want great events to attend)

If we only optimize for users (show them the most popular events), we create a **rich-get-richer** problem: popular hosts get more exposure, new hosts get none.

In [ ]:
# === Two-Sided Marketplace Fairness ===

def demonstrate_marketplace_fairness():
    """
    Show why optimizing only for users creates unfairness for hosts.
    
    12-year-old version:
    Imagine there are 100 event hosts. The top 10 are famous and
    everyone wants to go to their events. If we ONLY recommend
    popular events, the other 90 hosts get zero visibility.
    They stop creating events. The platform dies.
    
    We need to balance:
    - Showing users events they will love (user satisfaction)
    - Giving all hosts a fair chance to get discovered (host fairness)
    """
    print("=== Two-Sided Marketplace Problem ===")
    print()
    
    print("WITHOUT fairness:")
    print("  Top 10% of hosts get 90% of all impressions")
    print("  Bottom 50% of hosts get <1% of impressions")
    print("  New hosts get almost zero visibility")
    print("  Result: New hosts leave -> fewer events -> platform decline")
    print()
    
    print("WITH fairness mechanisms:")
    print("  1. New host boost: Give new hosts extra impressions initially")
    print("  2. Impression guarantee: Ensure every active host gets minimum impressions")
    print("  3. Category diversity: Show events from diverse hosts, not just popular ones")
    print("  4. Quality score: Rank by predicted quality, not just past popularity")
    print()
    
    print("=== Technical Solutions ===")
    solutions = {
        "Impression allocation": "Guarantee minimum impressions per host per day",
        "Boosting new hosts": "Multiply score by boost factor for hosts with < X events",
        "Exploration budget": "Reserve 10% of slots for underexposed events",
        "Multi-objective optimization": "Optimize for user_satisfaction + host_fairness jointly"
    }
    for solution, description in solutions.items():
        print(f"  {solution}: {description}")


demonstrate_marketplace_fairness()

## 6. Bias Considerations

In [ ]:
# === Types of Bias in Event Recommendation ===

print("=== Bias Types in Event Recommendation ===")
print()

biases = {
    "Position Bias": {
        "12yo": "Events shown at the top get more clicks, not because they are better, but because they are first.",
        "technical": "Users click items at position 1 much more than position 10, regardless of relevance.",
        "mitigation": "Inverse propensity weighting, randomized experiments, position-aware models."
    },
    "Popularity Bias": {
        "12yo": "Popular events get recommended more, which makes them MORE popular. Like a snowball effect.",
        "technical": "Feedback loop: popular items get more impressions -> more interactions -> even higher scores.",
        "mitigation": "Exploration, fairness constraints, popularity normalization."
    },
    "Demographic Bias": {
        "12yo": "If we use age and gender to recommend events, we might accidentally discriminate.",
        "technical": "Using protected attributes (age, gender) can create discriminatory recommendations.",
        "mitigation": "Fairness-aware models, remove protected attributes, counterfactual analysis."
    },
    "Selection Bias": {
        "12yo": "We only know about events users SAW. We do not know about events they NEVER saw.",
        "technical": "Training data is biased toward previously recommended items (missing-not-at-random).",
        "mitigation": "Causal inference, randomized exposure, inverse propensity scoring."
    },
    "Cold-Start Bias": {
        "12yo": "New events have no data, so they get recommended less, so they stay cold forever.",
        "technical": "New items with zero interactions get low scores, creating a vicious cycle.",
        "mitigation": "Content-based features, new item boosting, exploration budget."
    }
}

for bias, details in biases.items():
    print(f"--- {bias} ---")
    print(f"  Simple: {details['12yo']}")
    print(f"  Technical: {details['technical']}")
    print(f"  Fix: {details['mitigation']}")
    print()

In [ ]:
# === Feature Crossing for Expressiveness ===

print("=== Feature Crossing (Advanced Interview Topic) ===")
print()
print("Feature crossing creates NEW features by combining existing ones.")
print("This helps the model capture interactions that individual features cannot.")
print()

examples = {
    "age x category": {
        "raw": "age=25, category=Tech",
        "crossed": "age_25_AND_category_Tech = 1",
        "insight": "Young adults may prefer tech events differently than seniors"
    },
    "day_of_week x category": {
        "raw": "day=Saturday, category=Music",
        "crossed": "saturday_AND_music = 1",
        "insight": "Music events on weekends may have higher attendance"
    },
    "distance x price": {
        "raw": "distance=far, price=expensive",
        "crossed": "far_AND_expensive = 1",
        "insight": "Users may not travel far for cheap events but WILL for premium ones"
    },
    "friends_going x category": {
        "raw": "friends_going=3, category=Sports",
        "crossed": "friends_3_AND_sports = 1",
        "insight": "Social events (sports) with friends are much more appealing"
    }
}

for cross, details in examples.items():
    print(f"Cross: {cross}")
    print(f"  Raw features: {details['raw']}")
    print(f"  Crossed feature: {details['crossed']}")
    print(f"  Why: {details['insight']}")
    print()

print("Note: Neural networks can learn feature interactions implicitly,")
print("but explicit crosses help tree-based models (GBDT/XGBoost) significantly.")

## Key Takeaways

### Ranking
1. **Pointwise LTR** is our approach: binary classification for P(register)
2. Pairwise (RankNet, LambdaMART) and listwise (ListNet) are more accurate but harder
3. Start with GBDT, graduate to NN for continual learning

### Personalization
1. Four levels: popularity -> content-based -> feature-based -> deep personalization
2. Feature-based ranking (all 5 categories) is the core approach
3. Description similarity via TF-IDF captures content preferences

### Diversity
1. Without diversity, users get stuck in filter bubbles
2. MMR balances relevance and diversity with a tunable lambda
3. Category-aware re-ranking caps events per category
4. Epsilon-greedy exploration discovers new user interests

### Notifications
1. Budget per user per day to prevent fatigue
2. Quiet hours, relevance threshold, trigger-based decisions
3. Social triggers (friend signed up) are most powerful

### Advanced Topics
1. Two-sided marketplace: balance user satisfaction with host fairness
2. Five types of bias: position, popularity, demographic, selection, cold-start
3. Feature crossing for richer model expressiveness
4. Privacy considerations with location data and personal attributes

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)